In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import cv2
from PIL import Image
import numpy as np
import random
import os

In [ ]:
PATH = os.environ.get(
    "DATASET_PATH",
    "C:/Users/1/Desktop/BP/asl_dataset/asl_alphabet_train/asl_alphabet_train"
)

In [5]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 5
full_dataset_train = 15000
train_size = int(0.8 * full_dataset_train)
val_size = full_dataset_train - train_size

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [6]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])



val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

In [7]:
full_dataset_train = datasets.ImageFolder(PATH, transform=train_transform)
full_dataset_val   = datasets.ImageFolder(PATH, transform=val_transform)

indices = list(range(len(full_dataset_train)))
random.shuffle(indices)
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

train_set = torch.utils.data.Subset(full_dataset_train, train_indices)
val_set   = torch.utils.data.Subset(full_dataset_val,   val_indices)

In [8]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE)

In [9]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 8, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(8, 24, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(24, 48, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(48, 64, 3), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Flatten(),
            nn.Linear(2304, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 5),
        )


    def forward(self, x):
        return self.net(x)


model = Net().to(DEVICE)

print(model)

Net(
  (net): Sequential(
    (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 24, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(24, 48, kernel_size=(3, 3), stride=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(48, 64, kernel_size=(3, 3), stride=(1, 1))
    (10): ReLU()
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): Flatten(start_dim=1, end_dim=-1)
    (13): Linear(in_features=2304, out_features=512, bias=True)
    (14): ReLU()
    (15): Dropout(p=0.5, inplace=False)
    (16): Linear(in_features=512, out_features=256, bias=True)
    (17): ReLU()
    (18): Dropout(p=0.5, inplace=False)
    (19): Linear(in_features=256, out_features=5, bias=

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()

    acc = correct / val_size * 100
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.2f} | Val Accuracy: {acc:.2f}%")

Epoch 1/5 | Loss: 310.21 | Val Accuracy: 94.63%
Epoch 2/5 | Loss: 82.73 | Val Accuracy: 98.93%
Epoch 3/5 | Loss: 46.36 | Val Accuracy: 99.70%
Epoch 4/5 | Loss: 33.80 | Val Accuracy: 99.37%
Epoch 5/5 | Loss: 24.50 | Val Accuracy: 99.97%


In [11]:
torch.save(model.state_dict(), "asl_classifier/model.pth")
print("Model saved!")

Model saved!


In [ ]:
def run_webcam(model, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    CLASSES = ['A', 'B', 'C', 'D', 'nothing']

    inference_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    model.eval()
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        x1, y1, x2, y2 = 100, 100, 400, 400
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, "Put hand here", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        crop = frame[y1:y2, x1:x2]

        img = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        tensor = inference_transform(img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(tensor)
            pred = output.argmax(1).item()
            confidence = torch.softmax(output, dim=1)[0][pred].item()

        label = f"{CLASSES[pred]} ({confidence*100:.1f}%)"
        color = (0, 255, 0) if confidence > 0.7 else (0, 0, 255)
        cv2.putText(frame, label, (30, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, color, 3)

        cv2.imshow("ASL Classifier", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

In [17]:
run_webcam(model, device=DEVICE)